# CNN Model Tutorial

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/instadeepai/alf/blob/main/tutorials/models/cnn_tutorial.ipynb)

_Open in Colab works once ALF is public / on PyPI; until then, use the local setup below._

This tutorial covers how to use `CNNModel` in ALF for **regression** and **classification** tasks on protein sequences.

> **Prerequisites:** Familiarity with the ALF offline design loop.  
> See [offline_design_tutorial.ipynb](../experiments/offline_design_tutorial.ipynb) for that foundation.

### What you'll learn
1. How to configure and train `CNNModel` for regression (continuous output)
2. How to switch to binary classification (two phenotype classes)
3. How to switch to multiclass classification (three or more classes)
4. Which metrics are reported for each `ProblemType`

## Setup

Run the cell below to install ALF and this tutorial's dependencies — **no repository clone required**, so it works in a fresh environment or on Google Colab.

- Already set up a dev environment from a clone (`uv sync`)? You can **skip the install cell**.
- To run on a **GPU**, uncomment the GPU line in the install cell.

For all installation options, see the [Installation Guide](https://instadeepai.github.io/alf/installation.html).
- On **Colab**, the first install can take a few minutes; if prompted, choose *Runtime ▸ Restart session* and re-run the cell. To use a GPU, set *Runtime ▸ Change runtime type ▸ GPU* and uncomment the GPU line in the install cell.

In [ ]:
# Install ALF + this tutorial's dependencies — no clone needed.
# (Skip this cell if you are already running from a cloned repo via `uv sync`.)
# TODO(pypi): once ALF is published to PyPI, replace the git install below with:
#   %pip install alf_tools matplotlib
%pip install "alf_core @ git+https://github.com/instadeepai/alf.git#subdirectory=core" "git+https://github.com/instadeepai/alf.git#subdirectory=tools" matplotlib
# GPU (optional): run this AFTER the line above to switch PyTorch to a CUDA build.
# %pip install torch --index-url https://download.pytorch.org/whl/cu128

We import NumPy, the ALF candidate/dataset primitives, and the CNN model classes used throughout this tutorial:

In [ ]:
import numpy as np
from alf_core.dataclasses.candidate import Candidate, Modality
from alf_core.dataclasses.labelled_candidates import LabelledCandidates
from alf_core.dataset.base_dataset import BaseDataset, BaseDatasetConfig
from alf_core.utils.enums import ProblemType
from alf_tools.models import CNNModel, CNNModelConfig, CNNTrainConfig

print("Imports successful")

### What is a CNN surrogate, and when would you use one?

In ALF a *surrogate* is the cheap model that stands in for an expensive oracle (a wet-lab assay, a docking simulation) so the design loop can score many candidates per round. `CNNModel` is one such surrogate, specialised for **sequence** inputs such as proteins.

It works by one-hot encoding each sequence into an `(alphabet_size, seq_length)` matrix and passing it through a small stack of 1D convolutions followed by fully connected layers. The convolutions act as learnable motif detectors: each filter scans the sequence for a local amino-acid pattern, so the model can pick up short-range signal (binding motifs, local structure) without needing a hand-crafted featuriser. Unlike the `GPModel`, a CNN scales to large training sets and learns its own features, but it does **not** produce calibrated uncertainty — `predict()` returns point estimates only.

Reach for a CNN surrogate when you have enough labelled sequences to train a neural network and you care about predictive accuracy more than uncertainty estimates. This tutorial walks through the three problem types it supports — regression, binary classification, and multiclass classification — which differ only in the output layer and the metrics reported.

---
## Section 1: Regression

We use synthetic sequences with continuous fitness scores.  
`problem_type=ProblemType.REGRESSION` → single output neuron, MSE/Pearson/Spearman metrics.

In [ ]:
ALPHABET = list("ACDEFGHIKLMNPQRSTVWY")
SEQ_LEN = 20
rng = np.random.default_rng(42)


def random_sequences(n: int) -> list[str]:
    return ["".join(rng.choice(ALPHABET, SEQ_LEN)) for _ in range(n)]


def make_lc(seqs: list[str], labels: np.ndarray) -> LabelledCandidates:
    candidates = [Candidate(data=s, modality=Modality.SEQUENCE) for s in seqs]
    return LabelledCandidates(candidates=candidates, labels=labels)


seqs_reg = random_sequences(200)
labels_reg = rng.normal(loc=0.5, scale=0.2, size=200).clip(0, 1).astype(np.float32)

print(f"Sequences: {len(seqs_reg)}")
print(f"Label range: [{labels_reg.min():.3f}, {labels_reg.max():.3f}]")

We generate random 20-residue protein sequences with continuous fitness labels in `[0, 1]`. The labels here are pure noise rather than a learnable function, so do not expect high accuracy — the point of this tutorial is the *API and the shapes*, not model quality on a real signal.

Two ALF conventions appear in the helpers above. Each sequence is wrapped in a `Candidate` with `Modality.SEQUENCE` (the modality tells ALF the data is a string to be one-hot encoded), and the candidates plus their labels are bundled into a `LabelledCandidates`, which is the container every ALF model's `train()` expects.

In [ ]:
class SyntheticDataset(BaseDataset):
    def __init__(self, config: BaseDatasetConfig, lc: LabelledCandidates):
        super().__init__(config)
        self._lc = lc

    def load_dataset(self) -> LabelledCandidates:
        return self._lc

Build the regression dataset and split it. `problem_type=REGRESSION` is set on the **config**, not the model — `CNNModel` reads it from the dataset during `setup()`.

In [ ]:
reg_config = BaseDatasetConfig(
    name="synthetic_regression",
    modality=Modality.SEQUENCE,
    seed=42,
    train_ratio=0.8,
    validation_frac=0.2,
    test_ratio=0.2,
    split_type="random",
    problem_type=ProblemType.REGRESSION,
)

reg_dataset = SyntheticDataset(reg_config, make_lc(seqs_reg, labels_reg))
reg_dataset.setup()

Configure and train the CNN. For regression it ends in a single linear output neuron trained with MSE.

In [ ]:
cnn_regression = CNNModel(
    name="cnn_regression",
    model_config=CNNModelConfig(num_filters=32, kernel_size=3, num_conv_layers=2, fc_hidden_dim=64),
    train_config=CNNTrainConfig(num_epochs=5, batch_size=32, learning_rate=1e-3, log_frequency=5),
)

cnn_regression.setup(reg_dataset)
cnn_regression.train(
    train_data=reg_dataset.train_dataset,
    val_data=reg_dataset.validation_dataset,
)

print("Training complete.")
metrics = cnn_regression.get_training_summary_metrics()
print("Train metrics:", {k: f"{v:.4f}" for k, v in metrics.items()})

The cell above shows the full train-a-CNN workflow:

- The data is wrapped in a `BaseDataset`. Crucially, `problem_type` is declared on `BaseDatasetConfig`, not on the model — the model reads it from the dataset during `setup()`. `dataset.setup()` performs the train/validation/test split according to `train_ratio`, `validation_frac`, and `split_type`.
- `CNNModelConfig` controls the architecture: `num_filters` convolutional filters, a `kernel_size`-wide receptive field, `num_conv_layers` stacked conv blocks, and a `fc_hidden_dim`-wide fully connected head. `CNNTrainConfig` controls the optimisation (`num_epochs`, `batch_size`, `learning_rate`).
- `cnn.setup(dataset)` must be called before `train()`. It inspects the dataset's label space to size the output layer (here a single regression neuron) and validates the problem type. `train()` then runs the Adam training loop, reporting the metrics returned by `get_training_summary_metrics()` (for regression: MSE, Pearson, Spearman). With only five epochs on noise, these will be modest — that is expected.

In [ ]:
val_reg = reg_dataset.validation_dataset
preds = cnn_regression.predict(val_reg.candidates)
print(f"predictions.means shape: {preds.means.shape}")  # (n,)
print(f"predictions.variances:   {preds.variances}")  # None
print(f"Sample outputs: {preds.means[:5].round(4)}")

For regression, `predict()` returns a `Predictions` whose `means` has shape `(n,)` — one scalar per candidate — and whose `variances` is `None`, because a plain CNN gives point estimates with no uncertainty. (If you need uncertainty, use the `GPModel` or an ensemble surrogate instead.) The sample outputs are the model's predicted fitness for the first few validation sequences.

---
## Section 2: Binary Classification

Binary labels must be integers `{0, 1}`.  
`problem_type=ProblemType.BINARY` → single sigmoid output neuron expanded to a complement pair → `predictions.means` shape `(n, 2)`, where column 0 is P(class=0) = 1−σ(z) and column 1 is P(class=1) = σ(z).  
Metrics reported: `accuracy`, `f1`, `precision`, `recall`, `auc_roc`.

In [ ]:
seqs_bin = random_sequences(200)
labels_bin = (rng.random(200) > 0.5).astype(np.int32)

bin_config = BaseDatasetConfig(
    name="synthetic_binary",
    modality=Modality.SEQUENCE,
    seed=42,
    train_ratio=0.8,
    validation_frac=0.2,
    test_ratio=0.2,
    split_type="stratified",  # preserves class balance across splits
    problem_type=ProblemType.BINARY,
)

bin_dataset = SyntheticDataset(bin_config, make_lc(seqs_bin, labels_bin))
bin_dataset.setup()

print(f"num_classes: {bin_dataset.num_classes}")  # 2
print(f"Train: {len(bin_dataset.train_dataset)}, Val: {len(bin_dataset.validation_dataset)}")

Switching to classification is almost entirely a dataset-config change: we set `problem_type=ProblemType.BINARY` and give integer labels in `{0, 1}`. Note `split_type="stratified"` — for classification this preserves the class balance across the train/validation/test splits, so a rare class is not accidentally absent from a split. ALF derives `num_classes` automatically from the distinct integer labels (here, 2); you never set it by hand.

In [ ]:
cnn_binary = CNNModel(
    name="cnn_binary",
    model_config=CNNModelConfig(num_filters=32, kernel_size=3, num_conv_layers=2, fc_hidden_dim=64),
    train_config=CNNTrainConfig(num_epochs=5, batch_size=32, learning_rate=1e-3, log_frequency=5),
)

cnn_binary.setup(bin_dataset)
cnn_binary.train(
    train_data=bin_dataset.train_dataset,
    val_data=bin_dataset.validation_dataset,
)

print("Training complete.")
metrics = cnn_binary.get_training_summary_metrics()
print("Train metrics:", {k: f"{v:.4f}" for k, v in metrics.items()})

The construction and training code is identical to the regression case — same `CNNModelConfig`, same `setup()`/`train()` calls. Under the hood the model adapts to the problem type read from the dataset: it keeps a single output neuron but switches the loss to binary cross-entropy (`BCEWithLogitsLoss`) and reports classification metrics instead. This is the payoff of declaring `problem_type` on the dataset: the surrogate reconfigures itself, so the calling code does not change.

In [ ]:
val_bin = bin_dataset.validation_dataset
preds = cnn_binary.predict(val_bin.candidates)
print(f"predictions.means shape: {preds.means.shape}")  # (n, 2)
print("Sample class probabilities (first 3 rows):")
print(preds.means[:3].round(4))
print(f"Predicted classes: {preds.means[:3].argmax(axis=1)}")

For binary classification, `predict().means` has shape `(n, 2)`: the single sigmoid output is expanded into a complement pair, where column 0 is `P(class=0)` and column 1 is `P(class=1)`. Taking `argmax` over the two columns gives the predicted class. Returning both columns (rather than a single probability) keeps the output shape consistent with the multiclass case below, so downstream code can treat all classifiers uniformly.

---
## Section 3: Multiclass Classification

Multiclass labels must be integers `{0, 1, ..., K-1}` with at least 3 distinct classes.  
`problem_type=ProblemType.MULTICLASS` → softmax activation → `predictions.means` shape `(n, K)`.  
Metrics reported: `accuracy`, `f1` (macro), `precision` (macro), `recall` (macro), `auc_roc` (OvR).

In [ ]:
seqs_mc = random_sequences(300)
labels_mc = rng.integers(0, 4, size=300).astype(np.int32)  # 4 classes

mc_config = BaseDatasetConfig(
    name="synthetic_multiclass",
    modality=Modality.SEQUENCE,
    seed=42,
    train_ratio=0.8,
    validation_frac=0.2,
    test_ratio=0.2,
    split_type="stratified",
    problem_type=ProblemType.MULTICLASS,
)

mc_dataset = SyntheticDataset(mc_config, make_lc(seqs_mc, labels_mc))
mc_dataset.setup()

print(f"num_classes: {mc_dataset.num_classes}")  # 4

Multiclass works the same way, with `problem_type=ProblemType.MULTICLASS` and integer labels spanning three or more classes (here four, `{0, 1, 2, 3}`). As before, `num_classes` is inferred from the data — the printed value confirms ALF found four distinct labels.

In [ ]:
cnn_multiclass = CNNModel(
    name="cnn_multiclass",
    model_config=CNNModelConfig(num_filters=32, kernel_size=3, num_conv_layers=2, fc_hidden_dim=64),
    train_config=CNNTrainConfig(num_epochs=5, batch_size=32, learning_rate=1e-3, log_frequency=5),
)

cnn_multiclass.setup(mc_dataset)
cnn_multiclass.train(
    train_data=mc_dataset.train_dataset,
    val_data=mc_dataset.validation_dataset,
)

print("Training complete.")
metrics = cnn_multiclass.get_training_summary_metrics()
print("Train metrics:", {k: f"{v:.4f}" for k, v in metrics.items()})

The only structural change from binary classification is the output layer: the model now has `num_classes` output neurons trained with cross-entropy loss. The calling code is, again, unchanged. Reported metrics are the macro-averaged classification scores (`accuracy`, `f1`, `precision`, `recall`) plus one-vs-rest `auc_roc`.

In [ ]:
val_mc = mc_dataset.validation_dataset
preds = cnn_multiclass.predict(val_mc.candidates)
print(f"predictions.means shape: {preds.means.shape}")  # (n, 4)
print(f"Rows sum to 1 (softmax check): {preds.means[:3].sum(axis=1).round(4)}")
print(f"Predicted classes: {preds.means[:5].argmax(axis=1)}")

For multiclass, `predict().means` has shape `(n, K)` — a full softmax distribution over the `K` classes per candidate. Because it is a softmax, each row sums to 1 (the check above confirms this), and `argmax` over the row gives the predicted class. Note how regression, binary, and multiclass all return through the same `Predictions.means` field, only with different shapes: `(n,)`, `(n, 2)`, and `(n, K)` respectively. That uniformity is what lets the rest of the ALF pipeline stay agnostic to the problem type, which the summary table below makes explicit.

---
## Summary

| `problem_type` | Labels | Output shape | Activation | Metrics |
|---|---|---|---|---|
| `REGRESSION` | `float32` | `(n,)` | identity | MSE, Pearson, Spearman, pairwise_xent |
| `BINARY` | `int {0,1}` | `(n, 2)` | sigmoid → complement pair | accuracy, f1, precision, recall, auc_roc |
| `MULTICLASS` | `int {0…K-1}` | `(n, K)` | softmax | accuracy, f1 (macro), precision, recall, auc_roc (OvR) |

**Key points:**
- `problem_type` is set on `BaseDatasetConfig`, not on `CNNModel` directly — the model reads it from the dataset.
- `num_classes` is derived automatically from unique integer labels in the dataset.
- Use `split_type='stratified'` for classification to preserve class balance.
- `CNNModelConfig` is the same for all problem types — architecture does not change, only the output layer size and activation.